# Pneumonia Detection — 01: EDA & Preprocessing

Loads the chest X-ray dataset, explores class balance and sample images, and saves processed arrays to disk for use in the training notebook.

Data source: [Chest X-Ray Images (Pneumonia)](https://www.kaggle.com/datasets/paultimothymooney/chest-xray-pneumonia), Kermany et al., *Cell* (2018).

## Step 1: Imports

In [ ]:
import os
from glob import glob
import matplotlib.pyplot as plt
import random
import cv2
import pandas as pd
import numpy as np
import seaborn as sns
import skimage
from skimage.transform import resize
from tqdm import tqdm

from tensorflow.keras.utils import to_categorical

%matplotlib inline
import warnings
warnings.filterwarnings("ignore")

## Step 2: Download & load data

In [ ]:
# Run once per Colab session — requires your Kaggle API token set up first
# !pip install kaggle
# !kaggle datasets download -d paultimothymooney/chest-xray-pneumonia
# !unzip -q chest-xray-pneumonia.zip -d /content/chest_ray

In [ ]:
train_dir = "chest_ray/chest_xray/train/"
test_dir = "chest_ray/chest_xray/test/"

def get_data(folder):
    """
    Walks NORMAL/ and PNEUMONIA/ subfolders, reads each image,
    resizes to 150x150x3, and returns (X, y) numpy arrays.
    """
    X = []
    y = []
    for folderName in os.listdir(folder):
        if not folderName.startswith('.'):
            if folderName in ['NORMAL']:
                label = 0
            elif folderName in ['PNEUMONIA']:
                label = 1
            else:
                label = 2
            for image_filename in tqdm(os.listdir(folder + folderName)):
                img_file = cv2.imread(folder + folderName + '/' + image_filename)
                if img_file is not None:
                    img_file = skimage.transform.resize(img_file, (150, 150, 3))
                    img_arr = np.asarray(img_file)
                    X.append(img_arr)
                    y.append(label)
    X = np.asarray(X)
    y = np.asarray(y)
    return X, y

In [ ]:
X_train, y_train = get_data(train_dir)
X_test, y_test = get_data(test_dir)

print(f"X_train shape: {X_train.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"X_test shape: {X_test.shape}")
print(f"y_test shape: {y_test.shape}")

In [ ]:
# Encode labels to one-hot vectors (ex: 1 -> [0,1])
Y_trainHot = to_categorical(y_train, num_classes=2)
Y_testHot = to_categorical(y_test, num_classes=2)

## Step 3: Visualize data

Pixel values are scaled between 0 and 1 after resizing.

In [ ]:
def plotHistogram(a):
    """Plot histogram of RGB pixel intensities for a single image."""
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(a)
    plt.axis('off')
    histo = plt.subplot(1, 2, 2)
    histo.set_ylabel('Count')
    histo.set_xlabel('Pixel Intensity')
    n_bins = 30
    plt.hist(a[:, :, 0].flatten(), bins=n_bins, lw=0, color='r', alpha=0.5)
    plt.hist(a[:, :, 1].flatten(), bins=n_bins, lw=0, color='g', alpha=0.5)
    plt.hist(a[:, :, 2].flatten(), bins=n_bins, lw=0, color='b', alpha=0.5)

plotHistogram(X_train[1])

3 random X-rays from the Pneumonia class

In [ ]:
multipleImages = glob("/content/chest_ray/chest_xray/train/PNEUMONIA/*.jpeg")

def plotThreeImages(images):
    r = random.sample(images, 3)
    plt.figure(figsize=(16, 16))
    plt.subplot(131)
    plt.imshow(cv2.imread(r[0]))
    plt.subplot(132)
    plt.imshow(cv2.imread(r[1]))
    plt.subplot(133)
    plt.imshow(cv2.imread(r[2]))

plotThreeImages(multipleImages)

20 images — Normal vs Pneumonia comparison grids

In [ ]:
print("No Pneumonia")
multipleImages = glob('/content/chest_ray/chest_xray/train/NORMAL/*')
i_ = 0
plt.rcParams['figure.figsize'] = (10.0, 10.0)
plt.subplots_adjust(wspace=0, hspace=0)
for l in multipleImages[:25]:
    im = cv2.imread(l)
    im = cv2.resize(im, (128, 128))
    plt.subplot(5, 5, i_ + 1)
    plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    i_ += 1

In [ ]:
print("Yes Pneumonia")
multipleImages = glob('/content/chest_ray/chest_xray/train/PNEUMONIA/*')
i_ = 0
plt.rcParams['figure.figsize'] = (10.0, 10.0)
plt.subplots_adjust(wspace=0, hspace=0)
for l in multipleImages[:25]:
    im = cv2.imread(l)
    im = cv2.resize(im, (128, 128))
    plt.subplot(5, 5, i_ + 1)
    plt.imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
    plt.axis('off')
    i_ += 1

Class imbalance check

In [ ]:
map_characters = {0: 'No Pneumonia', 1: 'Yes Pneumonia'}

dfTrain = pd.DataFrame({"labels": y_train})
plt.figure(figsize=(6, 4))
sns.countplot(x="labels", data=dfTrain)
plt.xticks(ticks=[0, 1], labels=['No Pneumonia', 'Yes Pneumonia'])
plt.title("Training Set Class Distribution (Original, Imbalanced)")
plt.show()

print(map_characters)
print(dfTrain["labels"].value_counts())

## Step 4: Save processed arrays

Saves to disk so the training notebook (`02_training_evaluation.ipynb`) can load instantly instead of re-running `get_data()`, which is the slowest step in the pipeline.

In [ ]:
# If using Google Drive for persistence across sessions, mount it first:
# from google.colab import drive
# drive.mount('/content/drive')
# SAVE_DIR = '/content/drive/MyDrive/pneumonia_project/'

SAVE_DIR = './'  # change to your Drive path if desired

np.save(SAVE_DIR + 'X_train.npy', X_train)
np.save(SAVE_DIR + 'y_train.npy', y_train)
np.save(SAVE_DIR + 'X_test.npy', X_test)
np.save(SAVE_DIR + 'y_test.npy', y_test)

print("Saved: X_train.npy, y_train.npy, X_test.npy, y_test.npy")
print("Continue in 02_training_evaluation.ipynb")